# ROI classification: label, train, apply

1. Demix: `raw -> demixing_results.hdf5` (one file per session).
2. Label ROIs in the GUI, or apply a classifier you already have.
3. Train: the classifier is saved next to the hdf5 and its path is recorded in the file.
4. Use the labels / predictions downstream.

Every button in the GUI is also a method on `ClassificationVis`; `vis.wait()` blocks until
background work (loading, train, classify) has finished.

In [ ]:
import numpy as np
import h5py
import torch
from masknmf.visualization import ClassificationVis
from masknmf.classification import RoicatClassifier
from masknmf.demixing.demixing_results import DemixingResults

file_list = [r'X:\\data\\eunji\\masknmf-defaults\\zplane01\\demixing_results.hdf5']  # one demixing_results.hdf5 per session

## 1. Label

Keys `1-9` label the current ROI, `0` clears, up/down move, `u` jumps to the next unlabeled ROI, `h` opens help.
Labels autosave into each file: `DemixingResults/class_labels`, `label_names`, `roi_masks`.

In [ ]:
vis = ClassificationVis.from_masknmf(file_list, label_names=["soma", "dendrite", "junk"])
vis

### Labels from Python

In [ ]:
vis.wait()  # ROI images are built in the background
labels, names = vis.class_labels, vis.label_names  # (num_rois,) int64, -1 = unlabeled
print(f"{(labels >= 0).sum()}/{len(labels)} labeled", {n: int((labels == i).sum()) for i, n in enumerate(names)})

In [ ]:
vis.label([0, 1, 2], names.index("soma"))            # by ROI id
vis.goto(3)
vis.label_current(names.index("junk"))               # same as pressing 3 on ROI 3
vis.goto_next_unlabeled()
vis.add_label("axon")
vis.set_label_color(names.index("junk"), (0.9, 0.3, 0.9))
vis.class_labels_by_session                          # one array per file

In [ ]:
import matplotlib.pyplot as plt

labels, names = vis.class_labels, vis.label_names
fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 3))
for ax, (i, name) in zip(np.atleast_1d(axes), enumerate(names)):
    sel = labels == i
    ax.imshow(vis.roi_images[sel].mean(axis=0) if sel.any() else np.zeros(vis.roi_images.shape[1:]), cmap="gray")
    ax.set_title(f"{name} (n={sel.sum()})")
    ax.axis("off")

## 2. Train

Needs every ROI labeled, at least 2 per class. Same as the train button: saves to `vis.classifier_path`
(default `classifier.roicat_classifier` next to the first file) and records it in each hdf5
(`classifier_path`, `classifier_history`).

In [ ]:
vis.train()  # or vis.train("path/to/my_classifier")
vis.wait()
classifier_path = vis.classifier_path
print(vis.status)

Without the GUI (labels come from the files, or set `clf.labels` yourself):

In [ ]:
clf = RoicatClassifier.from_masknmf(file_list)  # picks up the labels stored in the files
clf.train(num_workers=0)
classifier_path = clf.save(classifier_path)      # also writes classifier.training.json: labels + training files

## 3. Apply to another session

Unlabeled ROIs take the prediction, existing labels are kept; the `pred` column shows name + confidence
(red where it disagrees with your label). Predictions are saved as `class_predictions`,
`class_probabilities` and `classified_with`.

In [ ]:
new_files = [NEW_r'X:\\data\\eunji\\masknmf-defaults\\zplane01\\demixing_results.hdf5']
vis2 = ClassificationVis.from_masknmf(new_files)
vis2.wait()
vis2.select_classifier(classifier_path)  # the Select classifier button
vis2.classify()                          # the classify button
vis2.wait()
print(vis2.status)
vis2

Without the GUI:

In [ ]:
clf = RoicatClassifier.from_disk(classifier_path)
ids, pred_names, probs = clf.classify(new_files, write=True)  # one entry per session; write=True stores them in the files
pred_names[0][:10], probs[0].max(axis=1)[:10]

## 4. Downstream

In [ ]:
with h5py.File(new_files[0], "r") as f:
    g = f["DemixingResults"]
    names = [n.decode() for n in g["label_names"][()]]
    labels = g["class_labels"][()]        # (num_rois,) int64, -1 = unlabeled
    masks = g["roi_masks"][()]            # (num_rois, 36, 36) float32
    preds = g["class_predictions"][()]    # (num_rois,) index into names
    conf = g["class_probabilities"][()]   # (num_rois,) confidence of preds
    print(g["classifier_path"][()].decode(), [h.decode() for h in g["classifier_history"][()]])

In [ ]:
dmr = DemixingResults.from_hdf5(new_files[0])  # labels ride along: dmr.class_labels, dmr.label_names
keep = dmr.roi_indices("soma")
traces = dmr.c[:, keep]                                      # (num_frames, num_soma)
footprints = dmr.a.index_select(1, torch.as_tensor(keep))    # (pixels, num_soma), sparse
traces.shape, footprints.shape

## Command line

`classification a.hdf5 [b.hdf5 ...] --labels soma,dendrite,junk [--classifier path]`

A `.npy` stack of `(num_rois, Y, X)` masks works too; its labels go to `<path>.labels.npz`.